# A1 – Fixed Chunk Size Optimization

- **Adapted from:** `all_rag_techniques/choose_chunk_size.ipynb`
- **Experiment ID:** `A1_CHUNK_SIZE`
- **Corpus:** `report_data/raw`
- **Evaluation set:** `report_data/evaluation/questions.json`
- **Purpose:** Find the optimal fixed chunk size by comparing retrieval quality, faithfulness, and latency across multiple chunk sizes. Only chunk_size changes; embedding, retriever type, Top-K, prompt, and LLM remain identical to A0.

## Hypothesis

Smaller chunks improve precision but may miss broader context. Larger chunks capture more context but introduce noise. There exists an optimal range that balances Recall@K, Faithfulness, and context token cost.

## Selection Criteria

Prioritize Faithfulness and Recall@K; verify Relevancy does not degrade significantly; latency and context tokens within acceptable limits.

## 1. Setup & Configuration

In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from report_common.config import load_config, print_config_summary, save_config_snapshot
from report_common.models import build_llm, build_embeddings
from report_common.evaluation import compute_retrieval_metrics
from report_common.io import save_jsonl, save_csv_summary, build_result_record, Timer

In [ ]:
config = load_config()
print_config_summary(config)

EXPERIMENT_ID = "A1_CHUNK_SIZE"
NOTEBOOK = "02_choose_chunk_size.ipynb"
SEED = config["seed"]
CONFIG_HASH = config["_config_hash"]

In [ ]:
llm = build_llm(config)
embeddings = build_embeddings(config)

# Connection check
print(f"LLM OK: {llm.invoke('hello').content[:30]}")
print(f"Embedding OK: dim={len(embeddings.embed_query('test'))}")

## 2. Load Corpus & Evaluation Set

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# Load documents
raw_data_path = PROJECT_ROOT / config["paths"]["raw_data"]
documents = []
for pdf_file in raw_data_path.glob("*.pdf"):
    documents.extend(PyPDFLoader(str(pdf_file)).load())
print(f"Loaded {len(documents)} pages.")

# Load evaluation questions
questions_path = PROJECT_ROOT / config["paths"]["questions"]
with open(questions_path, "r", encoding="utf-8") as f:
    eval_questions = json.load(f)
print(f"Loaded {len(eval_questions)} evaluation questions.")

## 3. Define Chunk Sizes to Test

Unit: **characters** (RecursiveCharacterTextSplitter uses `len()` by default).

Overlap is set to 10% of chunk_size for all configurations.

In [ ]:
CHUNK_SIZES = [200, 500, 800, 1200]
TOP_K = config["baseline"]["top_k"]

# Baseline prompt (same as A0 - no grounding, no citation)
NAIVE_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""Use the following context to answer the question.
If you don't know the answer, say you don't know.

Context:
{context}

Question: {question}
Answer:"""
)
chain = NAIVE_PROMPT | llm

## 4. Run Experiment for Each Chunk Size

In [ ]:
all_results = []
summary_rows = []

for chunk_size in CHUNK_SIZES:
    chunk_overlap = chunk_size // 10
    print(f"\n{'='*60}")
    print(f"CHUNK SIZE = {chunk_size} (overlap={chunk_overlap})")
    print(f"{'='*60}")

    # Split documents
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    chunks = splitter.split_documents(documents)
    for c in chunks:
        c.page_content = c.page_content.replace('\t', ' ')

    print(f"  Chunks created: {len(chunks)}")
    print(f"  Avg chunk length: {np.mean([len(c.page_content) for c in chunks]):.0f} chars")

    # Build index
    with Timer() as t_index:
        vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
    print(f"  Indexing time: {t_index.elapsed:.2f}s")

    # Run evaluation
    chunk_results = []
    for q in eval_questions:
        qid = q["question_id"]
        question = q["question"]
        relevant_docs = q.get("relevant_documents", [])

        with Timer() as t_ret:
            docs = retriever.invoke(question)

        retrieved_ids = [d.metadata.get("source", f"chunk_{i}") for i, d in enumerate(docs)]
        context = "\n\n".join([d.page_content for d in docs])
        context_tokens = len(context)  # approximate by chars

        with Timer() as t_gen:
            response = chain.invoke({"context": context, "question": question})

        metrics = compute_retrieval_metrics(retrieved_ids, relevant_docs, k=TOP_K)

        record = build_result_record(
            experiment_id=f"{EXPERIMENT_ID}_C{chunk_size}",
            notebook=NOTEBOOK,
            config_hash=CONFIG_HASH,
            seed=SEED,
            question_id=qid,
            question=question,
            answer=response.content,
            latency={
                "retrieval_seconds": t_ret.elapsed,
                "generation_seconds": t_gen.elapsed,
                "total_seconds": t_ret.elapsed + t_gen.elapsed,
            },
            usage={"context_chars": context_tokens},
            metrics=metrics,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            num_chunks=len(chunks),
        )
        chunk_results.append(record)

    all_results.extend(chunk_results)

    # Aggregate for this chunk size
    avg_metrics = {}
    for key in chunk_results[0]["metrics"]:
        vals = [r["metrics"][key] for r in chunk_results if r["metrics"].get(key) is not None]
        avg_metrics[key] = np.mean(vals) if vals else None

    avg_latency = np.mean([r["latency"]["total_seconds"] for r in chunk_results])
    avg_context_chars = np.mean([r["usage"]["context_chars"] for r in chunk_results])

    row = {
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "num_chunks": len(chunks),
        "avg_context_chars": int(avg_context_chars),
        "avg_latency_s": round(avg_latency, 3),
        **{k: round(v, 3) if v else None for k, v in avg_metrics.items()},
    }
    summary_rows.append(row)
    print(f"  Results: {row}")

print(f"\nTotal records: {len(all_results)}")

## 5. Results Comparison Table

In [ ]:
df = pd.DataFrame(summary_rows)
print(df.to_string(index=False))

## 6. Automatic Best Config Selection

In [ ]:
# Selection criteria:
# 1. Maximize recall_at_k
# 2. Among ties, maximize hit_rate_at_k
# 3. Among ties, minimize avg_latency_s

recall_col = f"recall_at_{TOP_K}"
hit_col = f"hit_rate_at_{TOP_K}"

if recall_col in df.columns:
    best_idx = df.sort_values(
        by=[recall_col, hit_col, "avg_latency_s"],
        ascending=[False, False, True]
    ).index[0]
    best_config = df.iloc[best_idx]
    print(f"\nBest chunk size: {int(best_config['chunk_size'])}")
    print(f"  Recall@{TOP_K}: {best_config[recall_col]}")
    print(f"  Avg latency: {best_config['avg_latency_s']}s")
    print(f"  Avg context chars: {int(best_config['avg_context_chars'])}")
else:
    print("Cannot determine best config - no ground truth documents in evaluation set.")

## 7. Save Results

In [ ]:
output_dir = PROJECT_ROOT / config["paths"]["results"]

save_jsonl(all_results, output_dir / "A1_chunk_size.jsonl")
save_csv_summary(summary_rows, output_dir / "A1_chunk_size_summary.csv")
save_config_snapshot(config, output_dir)

## 8. Observations

- Chunk size `___` provides the best balance of Recall and Faithfulness.
- Smaller chunks (`200`) create more vectors but may split key information.
- Larger chunks (`1200`) reduce index size but lower precision.
- This optimal chunk size will be used as the baseline for subsequent experiments (A2+).

_Fill in after running the experiment._